In [ ]:


import os

BASE_DIR = "/home/cecilia/Documentos/PIBIC/Rotulagem_Finetuning"

CAMINHO_DATASET = "/home/cecilia/Documentos/PIBIC/Rotulagem_Finetuning/subdataset.xlsx"

caminho_teste = os.path.join(BASE_DIR, "dados_parciais")
caminho_completo = os.path.join(BASE_DIR, "dados_finais.json")
caminho_excel_final = os.path.join(BASE_DIR, "excel_final.xlsx")
caminho_labels_treino = os.path.join(BASE_DIR, "labels_treino.json")
caminho_labels_teste = os.path.join(BASE_DIR, "labels_teste.json")

os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
import shutil
import re
import time
import json
import requests
import pandas as pd
from typing import List, Dict, Any
from tqdm.notebook import tqdm
from openai import OpenAI

In [ ]:
df = pd.read_excel(CAMINHO_DATASET)
print(f"{len(df)} reclamações carregadas.")

## Configuração da API (OpenRouter) e modelo professor


In [ ]:
api_key = ""
models = [
    'qwen/qwen-2.5-72b-instruct'
]

In [ ]:
system_prompt = (
"""
Persona:
Você é um especialista em análise de reclamações online, especificamente no CRM, responsável por desenvolver estratégias de negócios, com foco em entender as necessidades do consumidor.

Contexto:
    - O dataset contém reclamações extensas de consumidores online sobre diversos domínios
    - Cada reclamação contém pelo menos um alvo principal específico (podem ter mais de um), correspondente à frase específica a qual o cliente expressa o problema mais relevante da reclamação.
    - Termo de aspecto: Atributo específico ao qual a frase se refere.
    - Categoria do aspecto: Par no formato Entidade#Atributo, onde o 1º refere-se ao elemento da reclamação, e o 2º representa a dimensão que esta sendo avaliada.


Tarefa:
Você deve extrair de cada reclamação, um par contendo o aspecto e sua categoria.
Para isso você deve seguir os passos:
1- Encontre dentro da reclamação o alvo principal, frase a qual contém o problema central da reclamação. (podendo haver mais de um alvo)
2- Dentro do alvo (para cada alvo), encontre um par contendo o termo de aspecto e a sua respectiva categoria
3- Retorne na saída o alvo encontrado e embaixo o par solicitado.

Siga o modelo abaixo para a saída:
\tAlvo principal: o alvo principal da reclamação,
\tRótulo (s): aspecto|CATEGORIA

#########Atenção######:

Quando existir mais de um alvo por reclamação, retorne cada um com seu respectivo par abaixo, como no modelo a seguir:
        Alvo 1: \"1º alvo que você encontrar\"
        Rótulo(s): aspecto|CATEGORIA

        Alvo 2: \"2º alvo que você encontrar\"
        Rótulo(s): aspecto|CATEGORIA

Quando existir mais de um aspecto e categoria para o mesmo alvo, separe cada par por ';'. Siga este modelo:
\tRótulo(s): aspecto|CATEGORIA; aspecto|CATEGORIA

Quando o aspecto não estiver explicitamente escrito no texto, infera-o pelo contexto e escreva o Aspecto inferido e o termo 'implícito' entre '()'. Como no exemplo a seguir:
\tRótulo(s): Aspecto (implícito)|CATEGORIA

Se não for possível inferir nenhum aspecto, retorne somente a palavra 'implícito' no lugar do aspecto e a sua respectiva categoria:
\tRótulo(s): Implícito|CATEGORIA


#######Observações########:
- Separe cada par por uma barra vertical como esta: '|'
- Não extraia aspectos mencionados apenas como contexto, histórico ou consequência do problema principal. Priorize sempre o aspecto diretamente associado à reclamação central do consumidor.
- Não extraia aspectos secundários ou periféricos.
- Retorne somente as categorias em caixa alta. O termo de aspecto deve manter a capitalização natural do texto.
- Não utilize aspas (retas, curvas ou de qualquer tipo) ao redor do alvo ou do texto extraído. Escreva o texto diretamente, sem aspas envolvendo.


##########EXEMPLOS##############

######Exemplo1########

Reclamação completa:
\"Boa tarde Empréstimo consignado já foi pago documentos e contrato não consiste com vício documentos peço o cancelamento do contrato do banco inter\"

Alvo principal: Empréstimo consignado já foi pago
Rótulo(s): contrato|CONTRATO#CANCELAMENTO

######Exemplo2########

Reclamação completa:
\"Após o recebimento de ligação de cobrança, fui induzido a ingressar na plataforma do SERASA LIMPA NOME, na qual pude constatar a existência de um débito em meu nome no valor de R$ 3.084,11, correspondente ao contrato nº 1662873 débito este não reconhecido pelo NOTIFICANTE.\"

Alvo 1: constatar a existência de um débito em meu nome
Rótulo(s): débito|COBRANÇA#INDEVIDA

Alvo 2: débito este não reconhecido pelo NOTIFICANTE
Rótulo(s): débito|COBRANÇA#RECONHECIMENTO

######Exemplo3########

Reclamação completa:
\"Boa sorte para conseguir uma mesa.\"

Alvo principal: Boa sorte para conseguir uma mesa (implícito)
Rótulo(s): Implícito|RESTAURANTE#DIVERSOS

######Exemplo4########

Reclamação completa:
\"Ansioso por mais leituras dos mesmos autores.\"

Alvo 1: leituras
Rótulo(s): leituras|LIVRO#GERAL

Alvo 2: autores
Rótulo(s): autores|LIVRO#AUTOR

######Exemplo5########

Reclamação completa:
\"Comprei uma passagem para o voo AZUL2233 que deveria decolar às 14h, mas fomos informados apenas 20 minutos antes do horário previsto que o voo seria cancelado, sem nenhuma alternativa oferecida pela companhia até o momento.\"

Alvo principal: fomos informados apenas 20 minutos antes do horário previsto que o voo seria cancelado, sem nenhuma alternativa oferecida pela companhia até o momento

Rótulo(s): cancelamento|VOO#CANCELAMENTO; informação|ATENDIMENTO#COMUNICAÇÃO
"""
)

## Classe de inferência em lote (OpenRouter)

In [ ]:
class OpenRouterBatchInference:
    def __init__(self, api_key: str, models: List[str], system_prompt: str):

        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        self.models = models
        self.system_prompt = system_prompt

    def _create_messages(self, user_prompt: str) -> List[Dict[str, str]]:

        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Input: {user_prompt}\nOutput:"}
        ]

    def _query_model(self, model: str, user_prompt: str, tentativas=3) -> str:
        for tentativa in range(tentativas):
            try:
                completion = self.client.chat.completions.create(
                    model=model,
                    messages=self._create_messages(user_prompt),
                    timeout=120,
                    max_tokens=800,
                    extra_body={
                        "provider": {
                            "ignore": ["Novita"]
                        }
                    }
                )
                msg = completion.choices[0].message
                finish_reason = completion.choices[0].finish_reason

                if finish_reason == "length":
                    print("Resposta cortada por limite de tokens!")

                if msg.content:
                    return msg.content
                if hasattr(msg, 'reasoning') and msg.reasoning:
                    return msg.reasoning
                return "ERRO: output vazio"

            except Exception as e:
                print(f"Tentativa {tentativa+1}/{tentativas} falhou: {e}")
                time.sleep(5 * (tentativa + 1))

        return f"null|Error#API: falhou após {tentativas} tentativas"

    def generate_outputs(self, dataset: pd.DataFrame, save_path: str) -> Dict[str, List[Dict[str, Any]]]:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)

        all_outputs = {model: [] for model in self.models}

        if os.path.exists(save_path):
            with open(save_path, 'r', encoding='utf-8') as f:
                all_outputs = json.load(f)
                print("Progresso anterior carregado com sucesso!")

            for model in self.models:
                all_outputs[model] = [
                    item for item in all_outputs[model]
                    if not str(item.get("output", "")).startswith("null|Error#API")
                ]

        print("Montando os índices já processados na memória... Aguarde.")

        indices_processados = {
            model: set(
                int(item["index"]) for item in all_outputs[model]
                if "index" in item and not str(item.get("output", "")).startswith("null|Error#API")
            )
            for model in self.models
        }

        loop_progresso = tqdm(dataset.index, desc="Processando dataset")

        for index in loop_progresso:
            data_point = dataset.loc[index]
            user_prompt = f"Input: {data_point['texto_cliente']}"

            if all(int(index) in indices_processados[model] for model in self.models):
                continue

            for model in self.models:
                if int(index) in indices_processados[model]:
                    continue

                try:
                    loop_progresso.set_postfix(modelo=model)
                    output = self._query_model(model, user_prompt)

                    print(f"\n--- Reclamação {index} ---")
                    print(output)
                    print("-" * 50)

                    resultado = {
                        "index": int(index),
                        "input": data_point.to_dict(),
                        "output": output
                    }

                    all_outputs[model] = [
                        item for item in all_outputs[model]
                        if int(item["index"]) != int(index)
                    ]
                    all_outputs[model].append(resultado)
                    all_outputs[model].sort(key=lambda x: int(x["index"]))

                except Exception as e:
                    print(f"\n[Erro] no modelo {model} no índice {index}: {e}")

                    novo_resultado = {
                        "index": int(index),
                        "input": data_point.to_dict(),
                        "output": f"null|Error#API: {str(e)}"
                    }

                    all_outputs[model] = [
                        item for item in all_outputs[model]
                        if int(item.get("index", -1)) != int(index)
                    ]
                    all_outputs[model].append(novo_resultado)
                    all_outputs[model].sort(key=lambda x: int(x["index"]))

                time.sleep(1)

            with open(save_path, 'w', encoding='utf-8') as f:
                json.dump(all_outputs, f, ensure_ascii=False, indent=4)

            if (int(index) + 1) % 1000 == 0:
                caminho_backup = save_path.replace(".json", f"_backup_{int(index)+1}.json")
                with open(caminho_backup, 'w', encoding='utf-8') as f:
                    json.dump(all_outputs, f, ensure_ascii=False, indent=4)
                print(f"Backup salvo: {caminho_backup}")

        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_outputs, f, ensure_ascii=False, indent=4)

        return all_outputs

## Funções para contagem de erros e de créditos da API

In [ ]:
def contar_erros(caminho_json, modelo):
    with open(caminho_json, 'r', encoding='utf-8') as f:
        dados = json.load(f)
    erros = [item for item in dados[modelo] if str(item.get("output", "")).startswith("null|Error#API")]
    print(f"{len(erros)} reclamações com erro de API, serão reprocessadas na próxima execução.")
    return erros


def verificar_creditos(api_key):
    resp = requests.get(
        "https://openrouter.ai/api/v1/credits",
        headers={"Authorization": f"Bearer {api_key}"}
    )
    dados = resp.json()
    print(dados)
    return dados

In [ ]:
modelo = models[0]

inference = OpenRouterBatchInference(
    api_key=api_key,
    models=models,
    system_prompt=system_prompt
)

## Teste com 100 amostras  



In [ ]:
# Script para apagar um teste anterior e recomeçar do zero (opcional):
# if os.path.exists(caminho_teste): 
#     os.remove(caminho_teste)
#     print("Arquivo de teste antigo apagado, começando do zero.")

df_teste = df.head(100)
outputs_teste = inference.generate_outputs(df_teste, caminho_teste)

In [ ]:
contar_erros(caminho_teste, modelo)

## Rotulagem por blocos de 500 reclamações, pausando no indíce 9000 para uma conferência de créditos da API.


In [ ]:
# Script para apagar um resultado anterior e recomeçar do zero (opcional):
# if os.path.exists(caminho_completo):
#     os.remove(caminho_completo)
#     print("Arquivo anterior apagado, começando do zero.")

tamanho_bloco = 500
limite_sem_pausa = 9000

for inicio in range(0, len(df), tamanho_bloco):
    bloco = df.iloc[inicio : inicio + tamanho_bloco]
    print(f"Processando bloco {inicio} até {inicio + len(bloco) - 1}...")

    outputs_completo = inference.generate_outputs(bloco, caminho_completo)

    print("Bloco concluído. Confira seus créditos antes de continuar.")

    fim_do_bloco = inicio + tamanho_bloco
    if fim_do_bloco < limite_sem_pausa:
        print("Ainda dentro do limite sem pausa, seguindo direto pro próximo bloco...")
        continue

    input("Pressione Enter para continuar pro próximo bloco...")

In [ ]:
verificar_creditos(api_key)

## Controle de qualidade das extrações (identficação de extrações com recusas, truncamento por limite de tokens, erros de encoding e alucinação em outro idioma)




In [ ]:
with open(caminho_completo, 'r', encoding='utf-8') as f:
    outputs_completo = json.load(f)

In [ ]:
#identificação das recusas

def parece_recusa(texto):
    tem_pedido = bool(re.search(r'forne[cç]a|fornecer|colar\s+a\s+reclama|compartilhar\s+o\s+texto|acesso\s+ao\s+texto\s+completo', texto, re.IGNORECASE))
    tem_estrutura = bool(re.search(r'Alvo\s*(principal)?\s*\d*\s*:', texto, re.IGNORECASE))
    return tem_pedido and not tem_estrutura


def resposta_e_recusa(saida):
    return parece_recusa(saida)


indices_problematicos = []
for item in outputs_completo[modelo]:
    saida = item.get('output', '')
    if resposta_e_recusa(saida):
        indices_problematicos.append({
            "index": item["index"],
            "texto_original": item["input"]["texto_cliente"],
            "output_professor": saida
        })

print(f"{len(indices_problematicos)} reclamações confirmadas como recusa (sem falso positivo).")
for item in indices_problematicos:
    print("=" * 60)
    print("Index:", item["index"])
    print("Texto original:", item["texto_original"][:150])

In [ ]:

#identificação das extrações mal feitas por limite de token
def contar_tokens_aprox(texto):
    return len(texto) / 4  


max_tokens_usado = 800
margem_seguranca = 30

candidatos_truncados = []
for item in outputs_completo[modelo]:
    saida = item.get('output', '')
    tokens_estimados = contar_tokens_aprox(saida)
    if tokens_estimados >= (max_tokens_usado - margem_seguranca):
        candidatos_truncados.append({
            "index": item["index"],
            "tokens_estimados": round(tokens_estimados),
            "texto_original": item["input"]["texto_cliente"][:100],
            "output_professor": saida
        })

print(f"{len(candidatos_truncados)} reclamações próximas do limite de tokens (possível corte).")
for item in candidatos_truncados:
    print("=" * 60)
    print("Index:", item["index"], "| Tokens estimados:", item["tokens_estimados"])
    print("Final da resposta:", item["output_professor"][-150:])

In [ ]:

#Script para classificar as razões dos erros da extração
def tem_script_estrangeiro(texto):
    return bool(re.search(r'[\u4e00-\u9fff\uac00-\ud7af\u0400-\u04FF]', texto))


def tem_corrupcao_encoding(texto):
    return bool(re.search(r'Ã[£§©ª«¡\x80-\xFF]', texto))


def termina_realmente_cortado(texto):
    texto = texto.strip()
    if not texto:
        return True
    ultima_palavra = re.sub(r'[.,;:!?)\]}"]+$', '', texto.split()[-1]).lower()
    conectivos = {'e', 'de', 'da', 'do', 'que', 'com', 'para', 'por', 'como', 'em', 'no', 'na', 'a', 'o'}
    sem_pontuacao_final = texto[-1] not in '.!?)"}0123456789'
    return ultima_palavra in conectivos and sem_pontuacao_final


categorias = {"recusa": [], "encoding": [], "idioma_estrangeiro": [], "corte_real": []}

for item in outputs_completo[modelo]:
    saida = item.get('output', '')
    idx = item["index"]

    if parece_recusa(saida):
        categorias["recusa"].append(idx)
    elif tem_corrupcao_encoding(saida):
        categorias["encoding"].append(idx)
    elif tem_script_estrangeiro(saida):
        categorias["idioma_estrangeiro"].append(idx)
    elif termina_realmente_cortado(saida):
        categorias["corte_real"].append(idx)

for cat, indices in categorias.items():
    print(f"{cat}: {len(indices)} reclamações -> {indices}")

## Correção das extrações com erro de encoding

In [ ]:
!pip install -q ftfy

In [ ]:
indice_para_conferir_encoding = 797 #indíce com erro de encoding indetificado manualmente no dataset avaliado

import ftfy

# teste da correção em apenas um item 
for item in outputs_completo[modelo]:
    if item["index"] == indice_para_conferir_encoding:
        print("ANTES:", item["output"][:200])
        print("DEPOIS:", ftfy.fix_text(item["output"])[:200])

In [ ]:
# aplica a correção em todo o dataset
for item in outputs_completo[modelo]:
    item["input"]["texto_cliente"] = ftfy.fix_text(item["input"]["texto_cliente"])
    item["output"] = ftfy.fix_text(item["output"])

df['texto_cliente'] = df['texto_cliente'].apply(ftfy.fix_text)

with open(caminho_completo, 'w', encoding='utf-8') as f:
    json.dump(outputs_completo, f, ensure_ascii=False, indent=4)

## Remoção das recusas e reprocessamento dos índices com erro de corte por token e de alucinação de idioma.



In [ ]:
indices_recusa = [4840, 5175, 5462, 6047, 6657, 578] #indices com erro de recusa no dataset
indices_corte = [2535, 5342, 7636, 9066] #indices com corte por limite de token
indices_idioma = [159, 474, 612, 675, 852, 1020, 1168, 1392, 1543, 1608, 1666, 1889,
                  2031, 2113, 2147, 2358, 2517, 2795, 2840, 2968, 3261, 3288, 3682,
                  3689, 3746, 3754, 3891, 4057, 4342, 4451, 4569, 4604, 4730, 4734,
                  5027, 5323, 5340, 5727, 5728, 5768, 6303, 6687, 7147, 7252, 7339,
                  7855, 7950, 8009, 8056, 8186, 8326, 8627, 8695, 8699, 8832, 8940,
                  9093, 9139, 9526] #indices com alucinação de idioma da LLM 

indices_reprocessar = indices_corte + indices_idioma

# Remove as recusas permanentemente
outputs_completo[modelo] = [
    item for item in outputs_completo[modelo] if item["index"] not in indices_recusa
]

# Remoção das com erro de token e de alucinação
outputs_completo[modelo] = [
    item for item in outputs_completo[modelo] if item["index"] not in indices_reprocessar
]

with open(caminho_completo, 'w', encoding='utf-8') as f:
    json.dump(outputs_completo, f, ensure_ascii=False, indent=4)

print(f"Removidos permanentemente (recusa): {len(indices_recusa)}")
print(f"Removidos para reprocessar (corte+idioma): {len(indices_reprocessar)}")

In [ ]:
# Reprocessa os índices removidos por token e alucinação
df_reprocessar = df.loc[df.index.isin(indices_reprocessar)]
outputs_completo = inference.generate_outputs(df_reprocessar, caminho_completo)

In [ ]:
# Verificação se as extrações com recusa da LLM ainda estão no dataset
with open(caminho_completo, 'r', encoding='utf-8') as f:
    outputs_completo = json.load(f)

indices_presentes = set(item["index"] for item in outputs_completo[modelo])

print("Verificando se as reclamações de recusa ainda estão no arquivo:\n")
for idx in indices_recusa:
    status = "AINDA ESTÁ (erro!)" if idx in indices_presentes else "removida com sucesso"
    print(f"Index {idx}: {status}")

print(f"\nTotal de reclamações no arquivo agora: {len(indices_presentes)}")

## Geração do Excel (não usado no fine-tuning)

In [ ]:
def limpar_aspas(texto):
    if texto is None:
        return texto
    return texto.strip().strip('"').strip("'").strip()


def parse_output(texto):
    pares = []
    blocos = re.split(r'(?=Alvo\s*(?:principal)?\s*\d*\s*[:"])', texto, flags=re.IGNORECASE)

    for bloco in blocos:
        bloco = bloco.strip()
        if not bloco:
            continue

        alvo_match = re.search(r'Alvo[^:]*:\s*"?([^"\n]+)"?', bloco, flags=re.IGNORECASE)
        alvo = limpar_aspas(alvo_match.group(1)) if alvo_match else None

        rotulo_match = re.search(r'R[oó]tulo\(?s?\)?:?\s*"?(.+)', bloco, flags=re.IGNORECASE | re.DOTALL)
        rotulo = limpar_aspas(rotulo_match.group(1)) if rotulo_match else None

        if alvo and rotulo:
            pares.append((alvo, rotulo))

    return pares

In [ ]:
with open(caminho_completo, 'r', encoding='utf-8') as f:
    outputs_completo = json.load(f)

linhas = []
for item in outputs_completo[modelo]:
    indice = item["index"]
    frase = item["input"]["texto_cliente"]
    saida = item["output"]
    pares = parse_output(saida)

    if pares:
        for alvo, rotulo in pares:
            linhas.append({
                "index": indice,
                "Frase_Original": frase,
                "Alvo": alvo,
                "Rótulo": rotulo
            })
    else:
        linhas.append({
            "index": indice,
            "Frase_Original": frase,
            "Alvo": None,
            "Rótulo": saida
        })

df_final = pd.DataFrame(linhas)
df_final.to_excel(caminho_excel_final, index=False)
print(f"Salvo! {len(df_final)} linhas geradas.")

## Split do dataset em treino e teste para a utilização no fine-tuning.


In [ ]:
from sklearn.model_selection import train_test_split

indices_treino, indices_teste = train_test_split(
    df.index.tolist(), test_size=0.1, random_state=42
)

outputs_treino = {modelo: [item for item in outputs_completo[modelo] if item["index"] in indices_treino]}
outputs_teste  = {modelo: [item for item in outputs_completo[modelo] if item["index"] in indices_teste]}

with open(caminho_labels_treino, 'w', encoding='utf-8') as f:
    json.dump(outputs_treino, f, ensure_ascii=False, indent=4)

with open(caminho_labels_teste, 'w', encoding='utf-8') as f:
    json.dump(outputs_teste, f, ensure_ascii=False, indent=4)

print(f"Treino: {len(outputs_treino[modelo])} | Teste: {len(outputs_teste[modelo])}")